#### 문제
- GridSearchCV와 연동하기 위해서 Word2Vec class 생성한 것과 같이 
- 해당 class 보강 
- 모델을 선택할수 있도록 생성자 함수 추가적인 작업 
    - 3개의 매개변수를 추가
        - min_n(기본값 2), max_n(기본값 4), busket(기본값 2e+6)
    - 마지막 매개변수 1개 추가 
        - model을 선택할 수 있는 매개변수 
        - type의 기본값은 'w2v'
- fit함수 수정 
    - self.type에 따라서 학습이 되는 모델을 변경 
        - 'w2v' 라면 -> Word2Vec 학습하고 self.model에 대입
        - 'ft' 라면 -> FastText 학습하고 self.model에 대입

- 해당 클래스를 모듈화 
    - 모듈의 이름은 'gensim_test'

1. 모듈 로드 
2. tokenizer는 Okt 사용
3. 데이터 셋은 ratings_text.txt 파일을 로드 
4. 결측치 제거
5. 글자 간의 좌우 공백을 제거 
6. 빈 테스트 데이터가 document에 존재하는가? 제외 
7. 중복되는 document를 제외 
8. 상위 데이터 100개를 이용하여 gridsearch를 이용해서 파라미터 조합 
    - 파라미터 조합 (벡터화 : min_count은 1로 고정)
        - type :  ['w2v', 'ft']
        - vector_size : [80, 100]
    - 파라미터 조합 (학습 모델 : SVC)
        - C : [0.8, 1.0]
    - 계층화 폴드는 5회 
9. 하위 데이터 100개를 이용하여 검증 : 분류 레포트를 이용

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from konlpy.tag import Okt
import numpy as np 
import pandas as pd 
from gensim.models import Word2Vec, FastText
# sklearn에서 기본적으로 제공해주는 함수들을 상속 받기 위해서 특정 객체를 로드 
from sklearn.base import BaseEstimator, TransformerMixin
# 커스텀 모듈에서 class만 로드
from gensim_test import Vectorizer

In [30]:
df = pd.read_csv("../data/ratings_test.txt", sep='\t')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [ ]:
# 결측치 제외 
df.dropna(inplace=True)
# document에서 좌우의 공백을 제거 
df['document'] = df['document'].str.strip()

# 빈 텍스트 제외
df = df.loc[~(df['document'] == ''), ]

In [ ]:

# document의 중복 데이터를 제거
df.drop_duplicates('document', inplace = True)

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49157 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49157 non-null  int64 
 1   document  49157 non-null  object
 2   label     49157 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ MB


In [34]:
# 토큰화 함수 생성 
okt = Okt()

tokenizer = lambda x : [ word for word in okt.morphs(x) ]

In [35]:
pipe = Pipeline(
    [
        ('vector', Vectorizer(tokenizer=tokenizer, min_count=1)), 
        ('svc', SVC(random_state=42))
    ]
)

In [36]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [37]:
params = {
    'vector__type' : ['w2v', 'ft'], 
    'vector__vector_size' : [80, 100], 
    'vector__l2' : [False, True],
    'svc__C' : [0.8, 1.0]
}

In [38]:
grid = GridSearchCV(
    estimator= pipe, 
    param_grid= params, 
    cv = cv, 
    verbose=1
)

In [39]:
X_train = df.head(100)['document'].values
y_train = df.head(100)['label'].values
X_test = df.tail(100)['document'].values
y_test = df.tail(100)['label'].values

In [40]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
40 fits failed out of a total of 80.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1363, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ekfla\AppDat

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'svc__C': [0.8, 1.0], 'vector__l2': [False, True], 'vector__type': ['w2v', 'ft'], 'vector__vector_size': [80, 100]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,tokenizer,<function <la...001A306CC0720>


In [43]:
print("최적의 모델의 성능 점수 : ", grid.best_score_)

최적의 모델의 성능 점수 :  0.55


In [41]:
pred = grid.predict(X_test)

In [42]:
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

           0       0.78      0.60      0.68        67
           1       0.45      0.67      0.54        33

    accuracy                           0.62       100
   macro avg       0.62      0.63      0.61       100
weighted avg       0.67      0.62      0.63       100

